# 01 — Fingerprint (Butina) and physicochemical descriptor (k-means) clustering

Part of a 5-notebook peptide-clustering + consensus-split exploration:
`01_fingerprint_and_descriptor_clustering` -> `02_peptideclm_clustering` ->
`03_qmap_sequence_clustering` -> `04_consensus_comparison_and_split` ->
`05_diagnostics`. See `README_clustering.md` in this folder for the full picture and
recommended run order.

**What this notebook does.** Implements the first two of four independent,
CPU-only clustering "voters" over the 12,371 unique peptides in
`data/mic_classification_dataset.csv` (one row per organism the peptide was tested
against is deliberately collapsed to one row per `peptide_id` here — clustering a
peptide's chemistry doesn't depend on which organism it happened to be assayed
against):

- **Method (a):** Morgan/ECFP fingerprints + Tanimoto similarity + Butina clustering
  (RDKit) — the standard *structural* similarity view.
- **Method (b):** RDKit global physicochemical descriptors (MW, logP, TPSA, ...) +
  k-means — a *property-space* view, independent of exact substructure.

This notebook deliberately **ignores the dataset's existing `split` column** (per
the ask driving this whole exploration) and instead writes each method's per-peptide
cluster label into a shared on-disk table, `data/clustering/peptide_voters.parquet`,
that notebooks 02/03 add their own columns to and notebook 04 consumes for consensus.

Every clustering call lives in its own cell with parameters as named constants near
the top of that cell — rerun a cell with a different constant and only that method's
column changes; nothing upstream needs to be rerun.


In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator
from rdkit.ML.Cluster import Butina
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

RDLogger.DisableLog("rdApp.*")  # match src/soamp/features/peptide.py's convention


## Setup: locate the repo and load/build the shared peptide-level frame

Walk upward from the current working directory to find `pyproject.toml` rather than
hardcoding an absolute path (Jupyter's cwd is normally this notebook's own
directory, `scripts/EDA/`, but this makes every notebook robust to being run from
elsewhere too, e.g. `jupyter nbconvert --execute` from the repo root).


In [2]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root (no pyproject.toml found above cwd)")


REPO_ROOT = find_repo_root(Path.cwd())
DATA_DIR = REPO_ROOT / "data"
CLUSTERING_DIR = DATA_DIR / "clustering"
CLUSTERING_DIR.mkdir(exist_ok=True)
VOTERS_PATH = CLUSTERING_DIR / "peptide_voters.parquet"

print("REPO_ROOT:", REPO_ROOT)


REPO_ROOT: /Users/lukajin/PycharmProjects/soamp


In [3]:
def build_base_peptide_frame() -> pd.DataFrame:
    """One row per unique peptide_id, independent of the existing train/test `split`
    column (which this whole exploration deliberately ignores) and independent of
    which/how-many organisms the peptide was tested against.
    """
    classification_df = pd.read_csv(DATA_DIR / "mic_classification_dataset.csv")
    base = (
        classification_df
        .drop_duplicates(subset="peptide_id")[["peptide_id", "sequence", "smiles", "has_noncanonical"]]
        .reset_index(drop=True)
    )

    # bond_type isn't carried into mic_classification_dataset.csv (see
    # src/soamp/data/assembly.py's column list) but it *is* in the regression dataset
    # it was joined from -- pull it back in to derive is_linear, needed for the
    # linear-vs-non-linear coverage breakdown methods (c)/(d) report on later.
    regression_df = pd.read_csv(DATA_DIR / "final_mic_regression_dataset.csv")
    bond_types = regression_df.drop_duplicates(subset="peptide_id")[["peptide_id", "bond_type"]]
    base = base.merge(bond_types, on="peptide_id", how="left")
    base["is_linear"] = base["bond_type"].fillna("none") == "none"
    return base


def load_peptide_voters() -> pd.DataFrame:
    """Load the shared voter table, re-deriving the base columns fresh each time
    (so a dataset regeneration is always picked up) but preserving any voter columns
    another notebook already computed and saved.
    """
    base = build_base_peptide_frame()
    if VOTERS_PATH.exists():
        existing = pd.read_parquet(VOTERS_PATH)
        voter_cols = [c for c in existing.columns if c not in base.columns]
        base = base.merge(existing[["peptide_id", *voter_cols]], on="peptide_id", how="left")
    return base


def save_peptide_voters(df: pd.DataFrame) -> None:
    df.to_parquet(VOTERS_PATH, index=False)
    print(f"saved {VOTERS_PATH} ({len(df)} rows, columns: {list(df.columns)})")


peptide_voters = load_peptide_voters()
print(peptide_voters.shape)
peptide_voters.head()


(12371, 6)


,peptide_id,sequence,smiles,has_noncanonical,bond_type,is_linear
0,10,LFIFFF,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,False,none,True
1,11,RVKRVWPLVIRTVIAGYNLYRAIKKK,CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)[C@H](CCCN...,False,none,True
2,12,RKRIHIGPGRAFYTT,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1cnc[nH]1)NC(=O)...,False,none,True
3,13,RRXXRF,CC(=O)N[C@@H](CCCN=C(N)N)C(=O)N[C@@H](CCCN=C(N...,True,none,True
4,14,GIWDTIKSMGKVFAGKILQNL,CC[C@H](C)[C@H](NC(=O)CN)C(=O)N[C@@H](Cc1c[nH]...,False,none,True


In [4]:
def report_clustering_diagnostics(df: pd.DataFrame, cluster_col: str, runtime_seconds: float,
                                    imbalance_fraction_flag: float = 0.5) -> None:
    """Shared diagnostics printout for every voter method in this notebook set:
    cluster count, size distribution, a simple imbalance flag, coverage (% of
    peptides that got a non-null vote), and wall-clock runtime. Repeated (not
    imported) in each notebook so every notebook stays fully self-contained.
    """
    total = len(df)
    scored = df[cluster_col].notna()
    n_scored = int(scored.sum())
    sizes = df.loc[scored, cluster_col].value_counts()
    print(f"[{cluster_col}] runtime: {runtime_seconds:.2f}s")
    print(f"[{cluster_col}] coverage: {n_scored}/{total} peptides scored ({n_scored / total:.1%})")
    print(f"[{cluster_col}] cluster count: {len(sizes)}")
    print(f"[{cluster_col}] cluster size distribution: min={sizes.min()}, median={sizes.median():.0f}, "
          f"max={sizes.max()}, mean={sizes.mean():.1f}")
    largest_frac = sizes.max() / n_scored
    flag = " <-- SEVERE IMBALANCE" if largest_frac > imbalance_fraction_flag else ""
    print(f"[{cluster_col}] largest cluster is {largest_frac:.1%} of scored peptides{flag}")


## Method (a): Morgan/ECFP fingerprints + Tanimoto + Butina clustering

**Why this method / citation:** Butina clustering (D. Butina, *J. Chem. Inf. Comput.
Sci.* 1999, 39, 747-750) on a Tanimoto-distance matrix over Morgan (ECFP-equivalent)
fingerprints is RDKit's standard, textbook approach to structural clustering of a
compound library — it's a greedy, threshold-based clustering that doesn't require
picking a cluster *count* up front (unlike k-means), which suits chemical similarity
where cluster sizes are naturally very unequal.

Computed directly from `smiles` (never `sequence`) for every unique peptide, so
non-canonical/cyclic peptides are handled uniformly and precisely — matching this
project's existing SMILES-native philosophy in `src/soamp/features/peptide.py`.


In [5]:
# Adjustable parameters
MORGAN_RADIUS = 2       # ECFP4-equivalent
MORGAN_FPSIZE = 2048
BUTINA_DISTANCE_CUTOFF = 0.35  # Tanimoto DISTANCE cutoff (i.e. similarity >= 0.65 to
                                # co-cluster); 0.3-0.4 is a commonly used range for
                                # ECFP4 Butina clustering in the cheminformatics
                                # literature -- tune and rerun this cell freely.

_t0 = time.perf_counter()

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=MORGAN_FPSIZE)

smiles_list = peptide_voters["smiles"].tolist()
mols = [Chem.MolFromSmiles(s) if isinstance(s, str) and s else None for s in smiles_list]
parse_failed = [i for i, m in enumerate(mols) if m is None]
if parse_failed:
    print(f"WARNING: {len(parse_failed)} SMILES failed to parse -- these get a null vote, "
          f"not a guess (peptide_ids: {peptide_voters.loc[parse_failed, 'peptide_id'].tolist()[:10]}...)")

fps = [morgan_gen.GetFingerprint(m) if m is not None else None for m in mols]
valid_idx = [i for i, fp in enumerate(fps) if fp is not None]
valid_fps = [fps[i] for i in valid_idx]

# Butina.ClusterData wants the flat lower/upper-triangle of pairwise DISTANCES.
# This is O(n^2) in both time and memory (~76M floats at our ~12.4k scale, ~600MB as
# a Python list of floats) -- fast in wall-clock terms (RDKit's bulk similarity is
# implemented in C++) but worth knowing about if you scale this notebook up much
# further than the current dataset size.
n_valid = len(valid_fps)
dists = []
for i in range(1, n_valid):
    sims = DataStructs.BulkTanimotoSimilarity(valid_fps[i], valid_fps[:i])
    dists.extend(1.0 - s for s in sims)

butina_clusters = Butina.ClusterData(dists, n_valid, BUTINA_DISTANCE_CUTOFF, isDistData=True)

cluster_fingerprint = pd.Series(index=peptide_voters.index, dtype="float64")
for cluster_id, member_positions in enumerate(butina_clusters):
    for pos in member_positions:
        cluster_fingerprint.iloc[valid_idx[pos]] = cluster_id

peptide_voters["cluster_fingerprint"] = cluster_fingerprint
_runtime_a = time.perf_counter() - _t0

report_clustering_diagnostics(peptide_voters, "cluster_fingerprint", _runtime_a)


[cluster_fingerprint] runtime: 19.01s
[cluster_fingerprint] coverage: 12371/12371 peptides scored (100.0%)
[cluster_fingerprint] cluster count: 611
[cluster_fingerprint] cluster size distribution: min=1, median=2, max=1993, mean=20.2
[cluster_fingerprint] largest cluster is 16.1% of scored peptides


## Method (b): physicochemical descriptors + k-means

**Why this method / citation:** clustering on a small set of global physicochemical
descriptors (molecular weight, logP, TPSA, charge, rotatable bonds, ...) is standard
practice in QSAR/cheminformatics for grouping compounds by bulk property similarity
rather than substructure — a genuinely different signal from method (a)'s
fingerprint-based structural similarity.

Reuses `data/peptide_features.csv` (the 13 RDKit descriptors already computed by
`pipeline/features/01_build_peptide_features.py` / `src/soamp/features/peptide.py`
for every unique peptide) rather than recomputing them. Fits a **fresh**
`StandardScaler` here on all 12,371 peptides — deliberately *not* reusing the
pipeline's saved `data/peptide_feature_scaler.json`, which was fit only on the old
train split that this whole exploration is ignoring.


In [6]:
peptide_features = pd.read_csv(DATA_DIR / "peptide_features.csv")
descriptor_cols = [c for c in peptide_features.columns if c != "peptide_id"]

descriptor_df = peptide_voters[["peptide_id"]].merge(peptide_features, on="peptide_id", how="left")
missing_descriptors = descriptor_df[descriptor_cols].isna().any(axis=1)
if missing_descriptors.any():
    print(f"WARNING: {missing_descriptors.sum()} peptides missing precomputed descriptors -- null vote")

X = descriptor_df.loc[~missing_descriptors, descriptor_cols].values
X_scaled = StandardScaler().fit_transform(X)
print(X_scaled.shape)


(12371, 13)


In [7]:
# Small silhouette-score sweep to suggest a default k -- purely advisory, KMEANS_K
# below is the actual adjustable constant that decides what gets used.
_sweep_ks = [5, 10, 15, 20, 25, 30, 35, 40]
_sweep_scores = []
for k in _sweep_ks:
    labels = KMeans(n_clusters=k, random_state=42, n_init="auto").fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    _sweep_scores.append(score)
    print(f"k={k:>3}  silhouette={score:.4f}")

_suggested_k = _sweep_ks[int(np.argmax(_sweep_scores))]
print(f"\nsilhouette-suggested k: {_suggested_k}")


k=  5  silhouette=0.3407


k= 10  silhouette=0.2575


k= 15  silhouette=0.2483


k= 20  silhouette=0.2431


k= 25  silhouette=0.2444


k= 30  silhouette=0.2364


k= 35  silhouette=0.2226


k= 40  silhouette=0.2275

silhouette-suggested k: 5


In [8]:
KMEANS_K = _suggested_k  # override with any int to use a different k

_t0 = time.perf_counter()
kmeans_labels = KMeans(n_clusters=KMEANS_K, random_state=42, n_init="auto").fit_predict(X_scaled)

cluster_descriptor = pd.Series(index=peptide_voters.index, dtype="float64")
cluster_descriptor.loc[~missing_descriptors] = kmeans_labels
peptide_voters["cluster_descriptor"] = cluster_descriptor
_runtime_b = time.perf_counter() - _t0

report_clustering_diagnostics(peptide_voters, "cluster_descriptor", _runtime_b)


[cluster_descriptor] runtime: 0.01s
[cluster_descriptor] coverage: 12371/12371 peptides scored (100.0%)
[cluster_descriptor] cluster count: 5
[cluster_descriptor] cluster size distribution: min=92, median=2528, max=4604, mean=2474.2
[cluster_descriptor] largest cluster is 37.2% of scored peptides


## Save the two voter columns to the shared table

In [9]:
save_peptide_voters(peptide_voters)


saved /Users/lukajin/PycharmProjects/soamp/data/clustering/peptide_voters.parquet (12371 rows, columns: ['peptide_id', 'sequence', 'smiles', 'has_noncanonical', 'bond_type', 'is_linear', 'cluster_fingerprint', 'cluster_descriptor'])
